# Self-Representation Experiment — Full Pipeline

This notebook tests whether **LLaMA-3.1-8B has a dedicated internal representation of "self"** — a mechanistic interpretability question about whether the model encodes something like a self-concept in its weights.

The pipeline works by:
1. Generating prompts that ask the model to reason as different entity types (self, human, animal, object)
2. Recording the model's internal activations at every layer for each prompt
3. Finding a "self/other direction" in activation space that separates self-representation from others
4. Checking that this direction isn't just a surface-level confound (grammar, roleplay framing, animacy)
5. Visualising the results

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — make sure you selected a GPU runtime.')

## 1. Mount Google Drive

Colab sessions are temporary — when the session ends, everything in `/content/` is lost. Mounting Google Drive gives us a persistent location to save activations, directions, and figures so you don't have to re-run expensive steps if the session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ureca26_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Outputs will be saved to: {DRIVE_DIR}')

## 2. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ureca26.git'  # <-- update this
REPO_DIR = '/content/ureca26'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies (takes ~3 min on first run).
# Ignore numpy/jax/opencv conflict warnings — those are Colab pre-installed packages
# that don't affect this experiment.
!pip install -e . -q

## 3. HuggingFace login

LLaMA-3.1-8B is a gated model — you need a HuggingFace account with access approved.

**Recommended:** Store your token as a Colab Secret named `HF_TOKEN` (click the 🔑 key icon in the left sidebar) so you never paste it directly into the notebook.

If you don't have access yet: request it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option A: read from Colab Secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret.')
except Exception:
    # Option B: interactive prompt — will ask you to paste your token
    login()

## 4. Configure output paths

The default config in `configs/experiment.yaml` writes outputs to local paths like `data/` and `figures/`. Here we redirect all outputs to Google Drive so they persist across sessions. A temporary config file is written to `/tmp/` for this session.

In [ ]:
import yaml, os

CONFIG_PATH = 'configs/experiment.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['paths']['data_dir']        = f'{DRIVE_DIR}/data'
cfg['paths']['activations_dir'] = f'{DRIVE_DIR}/data/activations'
cfg['paths']['directions_dir']  = f'{DRIVE_DIR}/data/directions'
cfg['paths']['figures_dir']     = f'{DRIVE_DIR}/figures'
cfg['paths']['prompts_file']    = f'{DRIVE_DIR}/data/prompts.json'

COLAB_CONFIG = '/tmp/experiment_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

for d in cfg['paths'].values():
    if not d.endswith('.json'):
        os.makedirs(d, exist_ok=True)

print('Config written to', COLAB_CONFIG)
print('Output dirs created under', DRIVE_DIR)

## 5. Step 01 — Generate prompts

`01_generate_prompts.py` builds the dataset of text prompts used to probe the model.

It generates scenarios across **8 domains** (problem-solving, decision-making, social reasoning, task planning, self-assessment, knowledge boundaries, authority deference, tool use) for **5 entity classes** (self, expert human, average human, animal, object). Each prompt frames the same scenario from the perspective of one of these entities — e.g. *"As an AI assistant, how would you approach this problem?"* vs *"As a dog, how would you approach this problem?"*

Outputs:
- `data/prompts.json` — all prompts with metadata (entity class, domain, train/test split)
- `data/config_snapshot.yaml` — a copy of the config used

**Expected time:** < 1 minute (no model needed, pure Python)

In [ ]:
!python scripts/01_generate_prompts.py --config {COLAB_CONFIG}

## 6. Step 02 — Extract activations

`02_extract_activations.py` loads LLaMA-3.1-8B and runs every prompt through it, recording the **residual stream activations** at every layer (all 32 layers × 4096 hidden dimensions) for each prompt.

This is the core data collection step. Think of it as asking: *"what does the model's internal state look like at each layer when it's thinking about 'self' vs 'an animal'?"* We capture this at two token positions:
- **`final`** — the last token of the prompt (what the model is "thinking" right before generating a response)
- **`entity`** — the token corresponding to the entity word itself (e.g. "I", "dog", "expert")

Outputs (saved to `data/activations/`):
- `*_core.h5` — activations for the 200 main scenarios × 5 entity classes
- `*_ctrl_grammatical_person.h5` — control: first-person vs third-person phrasing
- `*_ctrl_role_play.h5` — control: roleplay framing
- `*_ctrl_animacy.h5` — control: living vs non-living entities

**Expected time: ~15–20 min on T4.** The model is ~16GB in float16, which fills T4 VRAM almost exactly — batch_size 4 is used to avoid OOM.

In [ ]:
!python scripts/02_extract_activations.py \
    --config {COLAB_CONFIG} \
    --batch_size 4
    # increase to 8 if running on A100/A10G

## 7. Step 03 — Find self/other directions

`03_find_directions.py` analyses the saved activations to find a **self/other direction** — a vector in the model's activation space that separates "thinking about self" from "thinking about other entities".

It does this two ways at each of the 32 layers:
- **Mean-difference direction** — the vector pointing from the average "other" activation to the average "self" activation
- **Logistic probe** — a linear classifier trained to predict entity class from activations, with cross-validated C hyperparameter tuning. The probe's weight vector is another candidate direction.

The best layer is chosen by probe test accuracy. It also computes pairwise cosine similarities between entity-class directions to see how geometrically distinct they are.

Outputs (saved to `data/directions/<model>/<token_position>/`):
- `direction_results.pkl` — directions, probe weights, accuracies, similarity matrices

**Expected time:** ~2–5 min (sklearn on CPU, no GPU needed)

In [ ]:
!python scripts/03_find_directions.py --config {COLAB_CONFIG}

## 8. Step 04 — Test specificity

`04_test_specificity.py` is the key validity check: **is the self/other direction actually about self-representation, or is it just picking up on surface-level features?**

It tests three confounds by measuring the cosine similarity between the self/other direction and each confound direction:
- **Grammatical person** — is the direction just detecting first-person ("I") vs third-person ("they") pronoun use?
- **Role-play** — is it just detecting roleplay framing in the prompt?
- **Animacy** — is it just detecting living vs non-living entities?

It also trains a **residual probe** — a classifier on activations with the self/other direction projected out — to check how much self-representation information remains after removing the direction. A large drop in accuracy = the direction captures most of the self-representation signal.

Bootstrap statistics (n=1000) are used to get confidence intervals on all similarity scores.

A good result: **low cosine similarity** with all three confounds, meaning the self/other direction is specific to self-representation rather than a surface artefact.

Outputs: `specificity_results.pkl`

**Expected time:** ~2–5 min

In [ ]:
!python scripts/04_test_specificity.py --config {COLAB_CONFIG}

## 9. Step 05 — Visualise results

`05_visualize.py` loads the direction and specificity results and generates publication-quality figures, including:
- **Probe accuracy by layer** — which layers best linearly separate entity classes
- **Pairwise direction similarity heatmap** — how geometrically distinct the entity-class directions are from each other
- **Confound cosine similarity by layer** — how much the self/other direction overlaps with grammatical person, animacy, and roleplay directions at each layer
- **Residual probe accuracy** — how much self-representation information survives after projecting out the self/other direction

Figures are saved to `figures/` on Google Drive and displayed inline below.

**Expected time:** < 1 min

In [ ]:
!python scripts/05_visualize.py --config {COLAB_CONFIG}

import glob
from IPython.display import Image, display

figs = sorted(glob.glob(f"{DRIVE_DIR}/figures/**/*.png", recursive=True))
print(f'Found {len(figs)} figures:')
for fig in figs:
    print(' ', fig)
    display(Image(fig))

## Done!

All outputs are saved to your Google Drive under `ureca26_outputs/` and will persist after this session ends:

```
ureca26_outputs/
├── data/
│   ├── prompts.json
│   ├── activations/   ← large HDF5 files (~GB)
│   └── directions/    ← probe weights, direction vectors
└── figures/           ← PNG plots
```

If you restart the session, you can skip steps 01–04 and jump straight to step 05 (or any later step) since the outputs are already saved to Drive.